# 13.8: Where the cost comes from

- **Input tokens are processed in prefill**, which is compute-bound and highly parallel (Section 3). A device can process tens of thousands of prompt tokens per second for a mid-sized model, using its arithmetic units efficiently.
- **Output tokens are produced in decode**, which is memory-bound. Even with large batches, the device produces far fewer output tokens per second, and each output token also occupies KV cache memory for the duration of the request.

A rough calculation for an 8B model on our hypothetical accelerator, priced at a hypothetical \$2.50 per hour:


In [ ]:
def cost_per_million(gpu_dollars_per_hour, tokens_per_second):
    tokens_per_hour = tokens_per_second * 3600
    return gpu_dollars_per_hour / tokens_per_hour * 1e6

gpu_price = 2.50                      # hypothetical $/hour for one accelerator
n_params = 8e9
# Prefill: compute-bound. Assume 50% of 1,000 TFLOP/s is achieved.
prefill_tps = 0.5 * 1e15 / (2 * n_params)
# Decode: batch of 128 at ~2,000 tokens of context; idealized 7,750 tok/s, assume half is achieved.
decode_tps = 0.5 * 7750

print(f"input  (prefill): {prefill_tps:>8,.0f} tok/s -> ${cost_per_million(gpu_price, prefill_tps):.3f} per M tokens")
print(f"output (decode):  {decode_tps:>8,.0f} tok/s -> ${cost_per_million(gpu_price, decode_tps):.3f} per M tokens")


**Expected output**

Output:

```text
input  (prefill):   31,250 tok/s -> $0.022 per M tokens
output (decode):     3,875 tok/s -> $0.179 per M tokens
```
